### 📈 How to Trade Options with the Black-Scholes Model

##### ▶️ Related Quant Guild Videos:

- [Quant Explains Risk-Neutral Option Pricing](https://youtu.be/wYpg0TGxvgM)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

- [How to Trade the Covered Call](https://youtu.be/iPsPRQlDeTA)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

In [ ]:
%%html
<style>
/* Overwrite the hard-coded white background for ipywidgets */
.cell-output-ipywidget-background {
    background-color: transparent !important;
}
/* Set widget foreground text and color to match the VS Code dark theme */
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
</style>

### 📖 Sections

#### 1.) 📉 Black-Scholes Model

- Relative Option Pricing and Implied Volatility

- 🔬 How a Price Path Is Actually Generated (GBM, one step at a time)

- Delta, $N(d_1)$, $N(d_2)$ and the Probability of "Making Money"

#### 2.) 💭 Closing Thoughts and Future Topics

---

##### What sense can we make of price if it's apples to oranges?

The price of an ATM 15 day GOOG call is \$11.00

The price of an ATM 15 day NVDA call is \$11.00

*Which is more expensive?*

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. Parameters & Data Generation ---
T = 1.0         # Time to maturity (1 year)
r = 0.05        # Risk-free rate

# Spot Prices (Actuals for the right chart)
S0_nvda = 214
S0_goog = 340

# Normalized Base for Left Chart (Spot and Strike rebased to 100)
S0_norm = 100
K_norm = 100    

# Volatility setup (GOOG has higher variance)
vol_nvda = 0.25
vol_goog = 0.50

n_steps = 100   # Number of simulation steps
dt = T / n_steps

def call_payoff(S, K):
    return np.maximum(S - K, 0)

# Generate Price Paths
np.random.seed(42) # Fixed seed for reproducible display
dates = pd.date_range(start='2025-01-01', periods=n_steps+1, freq='B')

# NVDA Path (Actual & Normalized)
rets_nvda = np.random.normal((r - 0.5 * vol_nvda**2) * dt, vol_nvda * np.sqrt(dt), n_steps)
path_nvda = S0_nvda * np.exp(np.cumsum(rets_nvda))
path_nvda = np.insert(path_nvda, 0, S0_nvda)
path_nvda_norm = (path_nvda / S0_nvda) * S0_norm

# GOOG Path (Actual & Normalized)
rets_goog = np.random.normal((r - 0.5 * vol_goog**2) * dt, vol_goog * np.sqrt(dt), n_steps)
path_goog = S0_goog * np.exp(np.cumsum(rets_goog))
path_goog = np.insert(path_goog, 0, S0_goog)
path_goog_norm = (path_goog / S0_goog) * S0_norm

# Dynamic boundaries for plotting (based on normalized data for the left side)
x_min_payoff = min(np.min(path_nvda_norm), np.min(path_goog_norm), K_norm) * 0.85
x_max_payoff = max(np.max(path_nvda_norm), np.max(path_goog_norm), K_norm * 1.1) * 1.05
y_max_payoff = call_payoff(x_max_payoff, K_norm) * 1.1

S_range = np.linspace(x_min_payoff, x_max_payoff, 200)
payoff_vals = call_payoff(S_range, K_norm)

# Colors
off_white = "#e0e0e0"
strike_color = "#ff4b4b"
nvda_color = "#00ff88"  # Green
goog_color = "#00d1ff"  # Blue

# --- 2. Figure Setup ---
# Enable secondary_y for the second column
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Normalized Call Payoff Diagram", "Underlying Price Simulation"),
    horizontal_spacing=0.15,
    specs=[[{"secondary_y": False}, {"secondary_y": True}]]
)

# Base traces (Order strictly matters for animation frames)

# 1. Payoff curve (Left)
fig.add_trace(go.Scatter(x=S_range, y=payoff_vals, line=dict(color=off_white, width=3), showlegend=False), row=1, col=1)
# 2. Strike line (Left - normalized)
fig.add_trace(go.Scatter(x=[K_norm, K_norm], y=[0, y_max_payoff], line=dict(color=strike_color, width=1, dash='dash'), showlegend=False), row=1, col=1)
# 3. NVDA normalized price line (Left)
fig.add_trace(go.Scatter(x=[S0_norm, S0_norm], y=[0, y_max_payoff], line=dict(color=nvda_color, width=2), mode='lines', showlegend=False), row=1, col=1)
# 4. GOOG normalized price line (Left)
fig.add_trace(go.Scatter(x=[S0_norm, S0_norm], y=[0, y_max_payoff], line=dict(color=goog_color, width=2), mode='lines', showlegend=False), row=1, col=1)

# 5. NVDA Strike line (Right, attached to primary y-axis)
fig.add_trace(go.Scatter(x=[dates[0], dates[-1]], y=[S0_nvda, S0_nvda], line=dict(color=nvda_color, width=1, dash='dash'), opacity=0.5, showlegend=False), row=1, col=2, secondary_y=False)
# 6. GOOG Strike line (Right, attached to secondary y-axis)
fig.add_trace(go.Scatter(x=[dates[0], dates[-1]], y=[S0_goog, S0_goog], line=dict(color=goog_color, width=1, dash='dash'), opacity=0.5, showlegend=False), row=1, col=2, secondary_y=True)
# 7. NVDA path (Right, primary y-axis) - ONLY THIS IN LEGEND
fig.add_trace(go.Scatter(x=[dates[0]], y=[S0_nvda], line=dict(color=nvda_color, width=2.5), mode='lines', name="NVDA Spot", showlegend=True), row=1, col=2, secondary_y=False)
# 8. GOOG path (Right, secondary y-axis) - ONLY THIS IN LEGEND
fig.add_trace(go.Scatter(x=[dates[0]], y=[S0_goog], line=dict(color=goog_color, width=2.5), mode='lines', name="GOOG Spot", showlegend=True), row=1, col=2, secondary_y=True)

# --- 3. Animation Frames & Slider Steps ---
frames = []
slider_steps = []

for i in range(n_steps + 1):
    # Normalized prices for the left chart
    curr_nvda_norm = path_nvda_norm[i]
    curr_goog_norm = path_goog_norm[i]
    
    frame_name = f"f{i}"
    
    # Create Frame updating all 8 traces
    frames.append(go.Frame(
        data=[
            go.Scatter(x=S_range, y=payoff_vals),                           # 1
            go.Scatter(x=[K_norm, K_norm], y=[0, y_max_payoff]),            # 2
            go.Scatter(x=[curr_nvda_norm, curr_nvda_norm], y=[0, y_max_payoff]), # 3
            go.Scatter(x=[curr_goog_norm, curr_goog_norm], y=[0, y_max_payoff]), # 4
            go.Scatter(x=[dates[0], dates[-1]], y=[S0_nvda, S0_nvda]),      # 5
            go.Scatter(x=[dates[0], dates[-1]], y=[S0_goog, S0_goog]),      # 6
            go.Scatter(x=dates[:i+1], y=path_nvda[:i+1]),                   # 7
            go.Scatter(x=dates[:i+1], y=path_goog[:i+1])                    # 8
        ],
        name=frame_name
    ))
    
    # Create Slider Step
    step = {
        "args": [
            [frame_name],
            {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}
        ],
        "label": str(i),
        "method": "animate"
    }
    slider_steps.append(step)

fig.frames = frames

# --- 4. Layout with Buttons and Slider ---
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=650, width=1200,
    margin=dict(t=100, b=150, r=50), 
    # Overlay the legend in the upper right corner of the right chart area
    legend=dict(
        x=0.97,
        y=0.95,
        xanchor="right",
        yanchor="top",
        bgcolor='rgba(30,30,30,0.8)',
        bordercolor='rgba(255,255,255,0.25)',
        borderwidth=1
    ), 
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [None, {"frame": {"duration": 20, "redraw": False}, "fromcurrent": True}]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Step: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

axis_style = dict(
    showgrid=True, gridcolor='rgba(255,255,255,0.1)',
    tickfont=dict(color=off_white), linecolor=off_white,
    zeroline=False, title_font=dict(color=off_white)
)

# Left Plot Axes (Normalized)
fig.update_xaxes(axis_style, range=[x_min_payoff, x_max_payoff], title_text="Relative Asset Price (Base 100)", row=1, col=1)
fig.update_yaxes(axis_style, range=[-5, y_max_payoff], title_text="Relative Payoff", row=1, col=1)

# Right Plot X-Axis
fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="Date", row=1, col=2)

# Right Plot Twin Y-Axes (Color-coded actuals)
fig.update_yaxes(axis_style, range=[np.min(path_nvda)*0.9, np.max(path_nvda)*1.1], title_text="NVDA Price ($)", title_font=dict(color=nvda_color), tickfont=dict(color=nvda_color), row=1, col=2, secondary_y=False)
fig.update_yaxes(axis_style, range=[np.min(path_goog)*0.9, np.max(path_goog)*1.1], title_text="GOOG Price ($)", title_font=dict(color=goog_color), tickfont=dict(color=goog_color), showgrid=False, row=1, col=2, secondary_y=True)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Supply and demand create an equilibrium price, and the Black-Scholes model looks to explain the way markets are currently pricing these contracts.


##### 📌 Implied Volatility의 정확한 정의

Black-Scholes 콜 가격 공식은 입력이 5개입니다 (배당 없는 유럽형 콜).

$$C_{BS}(S, K, T, r, \sigma) = S\,N(d_1) - K e^{-rT} N(d_2)$$

$$d_1 = \frac{\ln(S/K) + (r + \tfrac12\sigma^2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

$S, K, T, r$ 는 시장에서 바로 보이지만 $\sigma$ 는 보이지 않습니다. 그래서 **거꾸로** 풉니다.

> **Implied volatility** $\sigma_{imp}$ 는 $\;C_{BS}(S, K, T, r, \sigma_{imp}) = C_{market}\;$ 을 만족하는 $\sigma$ 이다.

$C_{BS}$ 는 $\sigma$ 에 대해 단조증가(vega $>0$)라서 해는 하나뿐이고, 아래 코드는 `brentq` 로 이 방정식의 근을 찾습니다.

**꼭 기억할 것**
- IV는 **옵션 하나**(특정 기초자산·행사가·만기)의 **시장가격**에서 나오는 숫자 하나입니다. 주가 경로에서 나오는 값이 아닙니다.
- IV는 "시장 가격을 BS 모델의 σ 단위로 환산한 값"입니다. 미래 실현 변동성을 맞힌다는 보장은 없습니다.
- 가격(\$)은 기초자산끼리 비교할 수 없지만, IV(연율화 %)는 비교할 수 있습니다.

**감 잡기: ATM 근사.** $K = S$ 이고 $r \approx 0$ 이면 $C \approx \frac{1}{\sqrt{2\pi}} S\sigma\sqrt{T} \approx 0.4\,S\,\sigma\sqrt{T}$ 이므로

$$\sigma \approx \frac{C}{0.4\, S\sqrt{T}} \qquad (\sqrt{15/365} = 0.2027)$$

| | $S$ | $C/S$ | 근사 IV |
|---|---|---|---|
| NVDA | 214 | 5.14% | $11 / (0.4 \cdot 214 \cdot 0.2027) \approx 63\%$ |
| GOOG | 340 | 3.24% | $11 / (0.4 \cdot 340 \cdot 0.2027) \approx 40\%$ |

똑같은 \$11이지만 NVDA에게는 주가의 5.1%, GOOG에게는 3.2%입니다. 그래서 **NVDA 옵션이 변동성 기준으로 훨씬 비쌉니다.** 아래 코드의 정확한 값은 NVDA 62.63%, GOOG 39.02%입니다.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm
from scipy.optimize import brentq

# --- 1. Parameters & IV Calculation ---
T_days = 15
T = T_days / 365.0  # Time to maturity in years
r = 0.04            # Risk-free rate (4%)

S0_nvda = 214.0
S0_goog = 340.0
market_price = 11.00 # ATM Call price for both

# Black-Scholes Call Price Formula
def bs_call(S, K, T, r, vol):
    d1 = (np.log(S / K) + (r + 0.5 * vol**2) * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

# Root finder to back out Implied Volatility
def implied_volatility(price, S, K, T, r):
    objective = lambda vol: bs_call(S, K, T, r, vol) - price
    return brentq(objective, 1e-4, 5.0) # Search between 0.01% and 500% IV

iv_nvda = implied_volatility(market_price, S0_nvda, S0_nvda, T, r)
iv_goog = implied_volatility(market_price, S0_goog, S0_goog, T, r)

print(f"Calculated NVDA IV: {iv_nvda:.2%}")
print(f"Calculated GOOG IV: {iv_goog:.2%}")

# --- 2. Data Generation (Monte Carlo Matrix) ---
n_paths = 50
n_steps = T_days
dt = T / n_steps

np.random.seed(42) # Fixed seed for reproducible display
# T is measured in calendar days (15/365), so each step is one calendar day
dates = pd.date_range(start=pd.Timestamp.today().normalize(), periods=n_steps+1, freq='D')

# Every path uses the SAME vol (the IV). Paths differ only through the random draws Z.
def generate_paths(S0, vol, n_paths, n_steps, dt, r):
    paths = np.zeros((n_steps + 1, n_paths))
    paths[0] = S0
    for i in range(1, n_steps + 1):
        Z = np.random.standard_normal(n_paths)
        paths[i] = paths[i-1] * np.exp((r - 0.5 * vol**2) * dt + vol * np.sqrt(dt) * Z)
    return paths

# Generate actual price paths
paths_nvda = generate_paths(S0_nvda, iv_nvda, n_paths, n_steps, dt, r)
paths_goog = generate_paths(S0_goog, iv_goog, n_paths, n_steps, dt, r)

# Colors and randomized opacities
off_white = "#e0e0e0"
nvda_color = "#00ff88"  # Green
goog_color = "#00d1ff"  # Blue
opacities = np.random.uniform(0.15, 0.70, n_paths)

# Static Y-axis limits so the animation doesn't jitter
y_min_nvda, y_max_nvda = np.min(paths_nvda) * 0.95, np.max(paths_nvda) * 1.05
y_min_goog, y_max_goog = np.min(paths_goog) * 0.95, np.max(paths_goog) * 1.05

# --- 3. Figure Setup ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"NVDA Simulated Paths (IV: {iv_nvda:.1%})", 
        f"GOOG Simulated Paths (IV: {iv_goog:.1%})"
    ),
    horizontal_spacing=0.10
)

# --- Base Traces (Strict Order Required for Frames) ---
# Trace 0: NVDA horizontal start line (Left)
fig.add_trace(go.Scatter(x=[dates[0], dates[-1]], y=[S0_nvda, S0_nvda], line=dict(color=off_white, width=1, dash='dash'), opacity=0.5, name="NVDA Spot"), row=1, col=1)
# Trace 1: GOOG horizontal start line (Right)
fig.add_trace(go.Scatter(x=[dates[0], dates[-1]], y=[S0_goog, S0_goog], line=dict(color=off_white, width=1, dash='dash'), opacity=0.5, name="GOOG Spot"), row=1, col=2)

# Traces 2 to 51: NVDA Paths (Left)
for i in range(n_paths):
    fig.add_trace(go.Scatter(x=[dates[0]], y=[S0_nvda], mode='lines', line=dict(color=nvda_color, width=1.5), opacity=opacities[i], showlegend=False), row=1, col=1)

# Traces 52 to 101: GOOG Paths (Right)
for i in range(n_paths):
    fig.add_trace(go.Scatter(x=[dates[0]], y=[S0_goog], mode='lines', line=dict(color=goog_color, width=1.5), opacity=opacities[i], showlegend=False), row=1, col=2)


# --- 4. Animation Frames & Slider Steps ---
frames = []
slider_steps = []

for k in range(n_steps + 1):
    frame_data = []
    
    # 0-1: Static horizontal lines
    frame_data.append(go.Scatter(x=[dates[0], dates[-1]], y=[S0_nvda, S0_nvda]))
    frame_data.append(go.Scatter(x=[dates[0], dates[-1]], y=[S0_goog, S0_goog]))
    
    # 2-51: NVDA Path updates
    for i in range(n_paths):
        frame_data.append(go.Scatter(x=dates[:k+1], y=paths_nvda[:k+1, i]))
        
    # 52-101: GOOG Path updates
    for i in range(n_paths):
        frame_data.append(go.Scatter(x=dates[:k+1], y=paths_goog[:k+1, i]))

    frame_name = f"f{k}"
    frames.append(go.Frame(data=frame_data, name=frame_name))
    
    step = {
        "args": [[frame_name], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}],
        "label": str(k),
        "method": "animate"
    }
    slider_steps.append(step)

fig.frames = frames

# --- 5. Layout with Buttons and Slider ---
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=600, width=1200,
    margin=dict(t=100, b=150, l=50, r=50), 
    legend=dict(x=0.5, y=-0.2, xanchor='center', orientation='h', bgcolor='rgba(0,0,0,0)'),
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                # Slightly slower duration (100ms) to cleanly render 100 simultaneous paths
                "args": [None, {"frame": {"duration": 100, "redraw": False}, "fromcurrent": True}] 
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Day: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

axis_style = dict(
    showgrid=True, gridcolor='rgba(255,255,255,0.1)',
    tickfont=dict(color=off_white), linecolor=off_white,
    zeroline=False, title_font=dict(color=off_white)
)

# Set static ranges and styling
fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="Date", row=1, col=1)
fig.update_yaxes(axis_style, range=[y_min_nvda, y_max_nvda], title_text="NVDA Price ($)", row=1, col=1)

fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="Date", row=1, col=2)
fig.update_yaxes(axis_style, range=[y_min_goog, y_max_goog], title_text="GOOG Price ($)", row=1, col=2)

fig.show()

###### ______________________________________________________________________________________________________________________________________

### 🔬 주가 경로는 정확히 어떻게 만들어지는가

위 그림의 NVDA 50개 경로는 모두 **같은 σ (= IV 62.63%) 하나**로 만들어졌습니다. 경로마다 다른 건 **난수 $Z$** 뿐입니다.
**경로마다 implied vol이 따로 있는 게 아닙니다.** 이 섹션에서는 경로 하나를 한 스텝씩 뜯어서 이걸 직접 확인합니다.

#### 1) 모델: 위험중립 측도 $\mathbb{Q}$ 에서의 기하 브라운 운동(GBM)

$$dS_t = r\,S_t\,dt + \sigma\,S_t\,dW_t^{\mathbb{Q}}$$

| 기호 | 의미 | 이 노트북에서 |
|---|---|---|
| $r$ | drift. 실제 기대수익률 $\mu$ 가 **아니라** 무위험이자율. 목적이 옵션 *가격* 계산이라 위험중립 측도에서 시뮬레이션하고, 거기서는 모든 자산의 기대수익률이 $r$ 입니다. | 4% |
| $\sigma$ | 변동성. **상수**. 시장가격에서 역산한 IV를 넣음. 모든 경로·모든 시점에서 같음. | 62.63% |
| $W_t$ | 브라운 운동. **무작위성의 유일한 원천** | 난수 $Z$ 로 구현 |

#### 2) 정확해 → 한 스텝 업데이트 식

이토 보조정리로 $\ln S_t$ 에 대해 풀면 다음 식이 나옵니다.

$$\boxed{S_{t+\Delta t} = S_t \exp\Big(\underbrace{(r - \tfrac12\sigma^2)\Delta t}_{\text{drift 항 (상수)}} + \underbrace{\sigma\sqrt{\Delta t}\,Z_t}_{\text{충격 항 (무작위)}}\Big), \qquad Z_t \sim N(0,1)\ \text{i.i.d.}}$$

오일러 근사가 아니라 **정확해**이므로 $\Delta t$ 가 하루든 1년이든 분포가 정확합니다.

- **$r\Delta t$**: 하루치 무위험 성장.
- **$-\tfrac12\sigma^2\Delta t$** (이토 보정): $e^X$ 는 볼록함수라서 $E[e^X] = e^{E[X] + \frac12 \mathrm{Var}(X)}$ 입니다. 이 항이 없으면 평균 주가가 $S_0e^{rT}$ 보다 커집니다. 이 항 덕분에 정확히 $E^{\mathbb{Q}}[S_T] = S_0 e^{rT}$ 가 됩니다 (아래 셀에서 확인).
- **$\sigma\sqrt{\Delta t}\,Z_t$**: 로그수익률의 무작위 부분. 분산이 시간에 비례하므로($\mathrm{Var} = \sigma^2\Delta t$) 표준편차는 $\sqrt{\Delta t}$ 에 비례합니다.

#### 3) 이 예제의 숫자

- $T = 15/365$년을 15스텝으로 나눔 → $\Delta t = 1/365$ (달력일 하루)
- 하루 로그수익률 표준편차 $= \sigma\sqrt{\Delta t} = 0.6263 \times \sqrt{1/365} \approx 3.28\%$
- 하루 drift $= (0.04 - \tfrac12 \cdot 0.6263^2)/365 \approx -0.043\%$ (이토 보정이 $r$ 보다 커서 음수)

#### 4) 만기 주가는 $Z$ 들의 **합**에만 의존

스텝을 모두 곱하면 지수 안의 항이 더해집니다.

$$S_T = S_0 \exp\Big((r-\tfrac12\sigma^2)T + \sigma\sqrt{\Delta t}\sum_{k=1}^{n} Z_k\Big) = S_0 \exp\Big((r-\tfrac12\sigma^2)T + \sigma\sqrt{T}\,Z^*\Big),\quad Z^* = \tfrac{1}{\sqrt n}\textstyle\sum_k Z_k \sim N(0,1)$$

그래서 만기 분포는 **로그정규분포**이고, $P(S_T > K)$ 를 닫힌 식으로 쓸 수 있습니다. 이게 뒤에 나오는 $N(d_2)$ 입니다.

In [ ]:
# ============================================================
# 🔬 Step 1. Dissect ONE price path, day by day
# ============================================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm
from scipy.optimize import brentq

# --- Inputs (same as the NVDA example above) ---
T_days = 15
T = T_days / 365.0
r = 0.04
S0 = 214.0
K = S0               # ATM
market_price = 11.00

def bs_call(S, K, T, r, vol):
    d1 = (np.log(S / K) + (r + 0.5 * vol**2) * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def implied_volatility(price, S, K, T, r):
    return brentq(lambda vol: bs_call(S, K, T, r, vol) - price, 1e-4, 5.0)

# The ONE volatility number used for every path and every step
sigma = implied_volatility(market_price, S0, K, T, r)

n_steps = T_days
dt = T / n_steps

drift_term = (r - 0.5 * sigma**2) * dt   # constant: identical for every day and every path
shock_scale = sigma * np.sqrt(dt)        # constant: identical for every day and every path

print(f"σ (IV, backed out of the $11 price) = {sigma:.4%}")
print(f"Δt                                  = {dt:.6f} years (= 1 calendar day)")
print(f"drift term  (r - σ²/2)·Δt           = {drift_term:+.6f}  ({drift_term:+.4%} per day)")
print(f"shock scale σ·√Δt                   = {shock_scale:.6f}  ({shock_scale:.4%} per day, 1 std-dev)")
print()

# --- The only random input: 15 standard normal draws ---
rng = np.random.default_rng(7)
Z = rng.standard_normal(n_steps)

rows = []
S = S0
for day in range(1, n_steps + 1):
    z = Z[day - 1]
    shock = shock_scale * z                 # random part of the log-return
    log_ret = drift_term + shock            # total log-return for the day
    S_new = S * np.exp(log_ret)             # exact GBM update
    rows.append({
        "day": day,
        "Z": z,
        "drift term": drift_term,
        "shock σ√Δt·Z": shock,
        "log-return": log_ret,
        "S (prev)": S,
        "S (new)": S_new,
        "daily % change": np.exp(log_ret) - 1,
    })
    S = S_new

steps = pd.DataFrame(rows).set_index("day")
display(steps.style.format({
    "Z": "{:+.3f}",
    "drift term": "{:+.5f}",
    "shock σ√Δt·Z": "{:+.5f}",
    "log-return": "{:+.5f}",
    "S (prev)": "{:.2f}",
    "S (new)": "{:.2f}",
    "daily % change": "{:+.2%}",
}))

# --- Check: S_T depends only on the SUM of the Z's ---
S_T_loop = steps["S (new)"].iloc[-1]
S_T_direct = S0 * np.exp(n_steps * drift_term + shock_scale * Z.sum())
Z_star = Z.sum() / np.sqrt(n_steps)
S_T_onestep = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z_star)
print(f"S_T from the day-by-day loop          : {S_T_loop:.6f}")
print(f"S_T from S0·exp(n·drift + σ√Δt·ΣZ)    : {S_T_direct:.6f}")
print(f"S_T in ONE step with Z* = ΣZ/√n = {Z_star:+.3f}: {S_T_onestep:.6f}")
print(f"→ {'ITM' if S_T_loop > K else 'OTM'} at maturity (K = {K:.0f})")

# --- Plot: price path (top) and the daily random shocks that drove it (bottom) ---
dates = pd.date_range(start=pd.Timestamp.today().normalize(), periods=n_steps + 1, freq='D')
path = np.concatenate([[S0], steps["S (new)"].values])

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35], vertical_spacing=0.08,
                    subplot_titles=("One NVDA path (σ = IV for every step)", "Daily log-return = drift term + σ√Δt·Z"))
fig.add_trace(go.Scatter(x=dates, y=path, mode='lines+markers', line=dict(color="#00ff88", width=2), name="S_t"), row=1, col=1)
fig.add_hline(y=K, line=dict(color="#e0e0e0", dash='dash', width=1), annotation_text="Strike", row=1, col=1)
fig.add_trace(go.Bar(x=dates[1:], y=steps["log-return"],
                     marker_color=np.where(steps["log-return"] > 0, "#00ff88", "#ff4b4b"), name="log-return"), row=2, col=1)
for k in (1, -1):
    fig.add_hline(y=drift_term + k * shock_scale, line=dict(color="#e0e0e0", dash='dot', width=1),
                  annotation_text=f"{'+' if k > 0 else '−'}1 std (σ√Δt)", row=2, col=1)
fig.update_layout(template="plotly_dark", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                  height=650, width=1100, showlegend=False)
fig.update_yaxes(title_text="NVDA Price ($)", row=1, col=1)
fig.update_yaxes(title_text="log-return", tickformat=".1%", row=2, col=1)
fig.show()

#### 5) 경로 여러 개: σ는 같고 $Z$ 만 다르다

이제 경로 5개를 만들어 봅니다. 난수 행렬 $Z$ 의 모양은 (스텝 수 × 경로 수)이고, **열 하나가 경로 하나**입니다.
각 경로에 쓰인 σ를 표로 찍어보면 모두 같은 값입니다.

In [ ]:
# ============================================================
# 🔬 Step 2. Several paths: same σ, different random draws
# ============================================================
n_show = 5
rng = np.random.default_rng(2026)
Z_mat = rng.standard_normal((n_steps, n_show))          # column i = random draws for path i

log_rets = drift_term + shock_scale * Z_mat              # same drift & scale for all columns
paths5 = S0 * np.exp(np.vstack([np.zeros(n_show), np.cumsum(log_rets, axis=0)]))

print("Random draws Z (first 5 days; each column is one path):")
display(pd.DataFrame(Z_mat[:5], index=pd.Index(range(1, 6), name="day"),
                     columns=[f"path {i+1}" for i in range(n_show)]).round(3))

summary5 = pd.DataFrame({
    "σ used":            [sigma] * n_show,                          # identical
    "drift term / day":  [drift_term] * n_show,                     # identical
    "ΣZ":                Z_mat.sum(axis=0),                          # different
    "Z* = ΣZ/√n":        Z_mat.sum(axis=0) / np.sqrt(n_steps),       # different
    "S_T":               paths5[-1],                                 # different
    "ITM?":              paths5[-1] > K,
}, index=[f"path {i+1}" for i in range(n_show)])
display(summary5.style.format({"σ used": "{:.4%}", "drift term / day": "{:+.6f}",
                               "ΣZ": "{:+.3f}", "Z* = ΣZ/√n": "{:+.3f}", "S_T": "{:.2f}"}))

colors = ["#00ff88", "#00d1ff", "#ffd166", "#ef476f", "#c77dff"]
fig = go.Figure()
for i in range(n_show):
    fig.add_trace(go.Scatter(x=dates, y=paths5[:, i], mode='lines+markers', line=dict(color=colors[i], width=2),
                             name=f"path {i+1}  (Z* = {Z_mat[:, i].sum() / np.sqrt(n_steps):+.2f})"))
fig.add_hline(y=K, line=dict(color="#e0e0e0", dash='dash', width=1), annotation_text="Strike")
fig.update_layout(template="plotly_dark", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                  height=500, width=1100, title=f"5 paths, all with σ = {sigma:.2%}: only Z differs",
                  yaxis_title="NVDA Price ($)", xaxis_title="Date")
fig.show()

#### 6) "경로마다 변동성이 달라 보이는" 이유: Realized volatility

경로 $i$ 의 **실현 변동성**(realized vol)은 그 경로에서 실제로 나온 일별 로그수익률 $x_{i,1},\dots,x_{i,n}$ 의 표본표준편차를 연율화한 값입니다.

$$\hat\sigma_i = \frac{1}{\sqrt{\Delta t}}\sqrt{\frac{1}{n-1}\sum_{k=1}^{n}(x_{i,k}-\bar x_i)^2}$$

| | Implied vol $\sigma_{imp}$ | Realized vol $\hat\sigma_i$ |
|---|---|---|
| 어디서 나오나 | 옵션 **가격** → BS 역산 | **경로** → 표본 통계량 |
| 개수 | 옵션 하나당 1개 | 경로 하나당 1개 |
| 시뮬레이션에서 역할 | **입력** | **출력** |
| 이 예제 | 62.63% (모든 경로 공통) | 경로마다 다름 |

모든 경로의 **참** 변동성은 σ 하나지만, 수익률 표본이 15개뿐이라 $\hat\sigma_i$ 에는 추정오차가 있습니다.
정규분포 표본표준편차의 상대 표준오차는 약 $1/\sqrt{2(n-1)}$ 이라서 $n=15$ 이면 약 18.9%, 즉 $62.6\% \pm 11.8\%p$ 정도 흩어집니다.

같은 $T$ 를 더 잘게 쪼개면(스텝 수 ↑) 모든 경로의 $\hat\sigma_i$ 가 σ 한 점으로 모입니다. **참 변동성이 하나뿐이라는 증거**입니다.
(참고: 경로마다, 시점마다 변동성이 정말로 달라지는 모델은 Heston 같은 **확률변동성 모델**입니다. Black-Scholes는 아닙니다.)

In [ ]:
# ============================================================
# 🔬 Step 3. Realized vol per path vs. the single input σ
# ============================================================
def simulate_paths(S0, sigma, r, T, n_steps, n_paths, rng):
    dt = T / n_steps
    Z = rng.standard_normal((n_steps, n_paths))
    log_rets = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    return S0 * np.exp(np.vstack([np.zeros(n_paths), np.cumsum(log_rets, axis=0)]))

def realized_vol(paths, dt):
    x = np.diff(np.log(paths), axis=0)          # daily log-returns, one column per path
    return x.std(axis=0, ddof=1) / np.sqrt(dt)  # annualized sample std-dev

n_paths = 250
rng = np.random.default_rng(42)
paths_15 = simulate_paths(S0, sigma, r, T, 15, n_paths, rng)        # 15 steps: one per day (as in the notebook)
paths_fine = simulate_paths(S0, sigma, r, T, 15 * 96, n_paths, rng)  # 1,440 steps: one per 15 minutes

rv_15 = realized_vol(paths_15, T / 15)
rv_fine = realized_vol(paths_fine, T / (15 * 96))

table = pd.DataFrame({
    "steps per path": [15, 15 * 96],
    "input σ (all paths)": [sigma, sigma],
    "mean realized vol": [rv_15.mean(), rv_fine.mean()],
    "min": [rv_15.min(), rv_fine.min()],
    "max": [rv_15.max(), rv_fine.max()],
    "std across paths": [rv_15.std(), rv_fine.std()],
    "theory ≈ σ/√(2(n−1))": [sigma / np.sqrt(2 * 14), sigma / np.sqrt(2 * (15 * 96 - 1))],
}, index=["daily steps", "15-min steps"])
display(table.style.format("{:.2%}", subset=table.columns[1:]))

bins = dict(start=0.20, end=1.05, size=0.01)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=(f"15 daily returns per path: realized vol scatters widely around σ",
                                    f"1,440 intraday returns per path: realized vol collapses onto σ"))
fig.add_trace(go.Histogram(x=rv_15, xbins=bins, marker_color="#00ff88", name="15 daily steps"), row=1, col=1)
fig.add_trace(go.Histogram(x=rv_fine, xbins=bins, marker_color="#00d1ff", name="1,440 intraday steps"), row=2, col=1)
for row in (1, 2):
    fig.add_vline(x=sigma, line=dict(color="#ff4b4b", width=2, dash='dash'), row=row, col=1,
                  annotation_text=f"input σ = IV = {sigma:.2%}", annotation_position="top right",
                  annotation_font_color="#ff4b4b")
    fig.update_yaxes(title_text="Number of paths", row=row, col=1)
fig.update_xaxes(tickformat=".0%", range=[0.2, 1.05])
fig.update_xaxes(title_text="Realized volatility (annualized)", row=2, col=1)
fig.update_layout(template="plotly_dark", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                  height=650, width=1100, showlegend=False, bargap=0.05,
                  title=f"Realized vol of {n_paths} paths, all simulated with the same σ")
fig.show()

#### 7) 이토 보정 $-\tfrac12\sigma^2$ 은 왜 꼭 필요한가

위험중립 세계에서는 주식을 들고 있어도 평균적으로 무위험이자율만큼만 벌어야 합니다. 즉 $E^{\mathbb{Q}}[S_T] = S_0 e^{rT}$ 이어야 합니다.
아래 셀은 보정이 있을 때와 없을 때를 비교합니다. 만기 주가는 $Z^*$ 하나로 바로 뽑을 수 있으므로(4번 참고) 경로 전체 대신 만기 값만 200만 개 뽑습니다.

In [ ]:
# ============================================================
# 🔬 Step 4. Why the -σ²/2 term: E^Q[S_T] must equal S0·e^{rT}
# ============================================================
rng = np.random.default_rng(0)
Z_star = rng.standard_normal(2_000_000)

S_T_with = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z_star)   # correct
S_T_without = S0 * np.exp(r * T + sigma * np.sqrt(T) * Z_star)                   # WRONG: no Itô correction

target = S0 * np.exp(r * T)
print(f"Target  S0·e^(rT)                       = {target:.3f}")
print(f"Mean S_T WITH  -σ²/2 (correct)           = {S_T_with.mean():.3f}")
print(f"Mean S_T WITHOUT -σ²/2 (wrong)           = {S_T_without.mean():.3f}   "
      f"(theory: S0·e^(rT + σ²T/2) = {S0 * np.exp(r * T + 0.5 * sigma**2 * T):.3f})")
print()
print(f"Median S_T (correct) = {np.median(S_T_with):.3f}  <  mean {S_T_with.mean():.3f}")
print("→ The log-normal distribution is right-skewed: the median path ends BELOW S0·e^(rT),")
print("  a few large up-moves pull the mean up. That is also why P(S_T > K) is below 50% for an ATM strike.")

###### ______________________________________________________________________________________________________________________________________

##### Delta는 "대략" ITM 확률일 뿐, 정확히는 다르다: $N(d_1)$ vs $N(d_2)$

배당 없는 유럽형 콜에 대해 Black-Scholes 가정(σ 상수, 로그정규 주가)에서 정확한 정의를 정리하면 다음과 같습니다.

| 양 | 정확한 정의 | 식 | 이 예제 (NVDA) |
|---|---|---|---|
| **Delta** $\Delta$ | 콜 가격의 주가 민감도 $\partial C/\partial S$. **헤지 비율**: 콜 1개를 헤지하려면 주식 $\Delta$ 주를 매도 | $N(d_1)$ | 0.5305 |
| **위험중립 ITM 확률** | $\mathbb{Q}(S_T > K)$. 주가 drift $= r$ | $N(d_2)$ | **0.4798** |
| **주식측도 ITM 확률** | $\mathbb{Q}^S(S_T > K)$. 주식을 기준재(numeraire)로 삼는 측도. 주가 drift $= r+\sigma^2$ | $N(d_1)$ | 0.5305 |
| **실제(물리적) ITM 확률** | $\mathbb{P}(S_T > K)$. 주가 drift $=$ 실제 기대수익률 $\mu$ | $N(d_2^{\mu})$, $\;d_2^{\mu}=\dfrac{\ln(S/K)+(\mu-\frac12\sigma^2)T}{\sigma\sqrt T}$ | $\mu$ 에 따라 다름 |

**왜 헷갈릴까?** 콜 가격 공식을 두 조각으로 나눠 보면

$$C = \underbrace{S\,N(d_1)}_{S\,\cdot\,\mathbb{Q}^S(S_T>K)} \;-\; \underbrace{K e^{-rT}\,N(d_2)}_{Ke^{-rT}\,\cdot\,\mathbb{Q}(S_T>K)}$$

$N(d_1)$ 과 $N(d_2)$ 는 **둘 다 "ITM 확률"이지만 서로 다른 측도에서의 확률**입니다. delta와 같은 건 $N(d_1)$ (주식측도) 쪽입니다.
**이 노트북의 시뮬레이션은 drift $= r$ (위험중립 $\mathbb{Q}$) 이므로 ITM 비율은 $N(d_2) \approx 48\%$ 로 수렴합니다. delta 53%로 수렴하지 않습니다.**

**차이는 얼마나 클까?** $d_1 - d_2 = \sigma\sqrt T$ 입니다. 이 예제에서는 $0.6263 \times 0.2027 = 0.127$ 이라 두 확률이 약 5%p 차이 납니다.
만기가 짧고 변동성이 낮을수록 $\sigma\sqrt T \to 0$ 이 되어 "delta ≈ 확률" 이 근사적으로 맞고, 만기가 길거나 변동성이 높으면 크게 틀립니다.

**ATM인데 왜 50%가 아닐까?** $K=S$ 이면 $d_1 = \frac{(r+\frac12\sigma^2)T}{\sigma\sqrt T} > 0$ 이라서 delta는 50%보다 크고, $d_2 = \frac{(r-\frac12\sigma^2)T}{\sigma\sqrt T}$ 는 $\sigma^2/2 > r$ 이면 음수라서 위험중립 확률은 50%보다 작습니다. 위 셀에서 본 오른쪽 꼬리(평균 > 중앙값)와 같은 이야기입니다.

**마지막 함정:** $N(d_2)$ 조차 **위험중립** 확률이지 실제로 돈을 벌 확률이 아닙니다. 실제 확률은 $\mu$ 를 알아야 하는데, 옵션 가격에는 $\mu$ 가 들어있지 않습니다.

In [ ]:
# ============================================================
# Verify every row of the table above numerically
# ============================================================
sqT = np.sqrt(T)
d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqT)
d2 = d1 - sigma * sqT

# 1) Delta by definition: bump the stock price and reprice (central finite difference)
h = 0.01
delta_fd = (bs_call(S0 + h, K, T, r, sigma) - bs_call(S0 - h, K, T, r, sigma)) / (2 * h)

# 2) Probabilities by Monte Carlo on terminal prices, changing ONLY the drift (i.e. the measure)
rng = np.random.default_rng(1)
Z_star = rng.standard_normal(2_000_000)
def p_itm_mc(drift):
    S_T = S0 * np.exp((drift - 0.5 * sigma**2) * T + sigma * sqT * Z_star)
    return (S_T > K).mean()

rows = [
    ("Delta ∂C/∂S (finite difference)", "N(d1)", norm.cdf(d1), delta_fd),
    ("Q(S_T > K)   risk-neutral, drift r", "N(d2)", norm.cdf(d2), p_itm_mc(r)),
    ("Q^S(S_T > K) stock measure, drift r+σ²", "N(d1)", norm.cdf(d1), p_itm_mc(r + sigma**2)),
]
for mu in (0.00, 0.15, 0.30):
    d2_mu = (np.log(S0 / K) + (mu - 0.5 * sigma**2) * T) / (sigma * sqT)
    rows.append((f"P(S_T > K)   real world, μ = {mu:.0%}", "N(d2^μ)", norm.cdf(d2_mu), p_itm_mc(mu)))

check = pd.DataFrame(rows, columns=["quantity", "formula", "closed form", "numerical check"]).set_index("quantity")
print(f"d1 = {d1:+.4f},  d2 = {d2:+.4f},  d1 - d2 = σ√T = {sigma * sqT:.4f}")
display(check.style.format({"closed form": "{:.4f}", "numerical check": "{:.4f}"}))

# How the gap N(d1) - N(d2) grows with maturity (ATM, same σ)
horizons = {"7 days": 7/365, "15 days": 15/365, "30 days": 30/365, "90 days": 90/365, "1 year": 1.0, "2 years": 2.0}
gap = []
for label, tau in horizons.items():
    a = (r + 0.5 * sigma**2) * tau / (sigma * np.sqrt(tau))
    b = a - sigma * np.sqrt(tau)
    gap.append((label, sigma * np.sqrt(tau), norm.cdf(a), norm.cdf(b), norm.cdf(a) - norm.cdf(b)))
gap = pd.DataFrame(gap, columns=["maturity", "σ√T", "Delta N(d1)", "Q-prob N(d2)", "gap"]).set_index("maturity")
print("\nATM, σ = IV: the 'delta ≈ probability' shortcut gets worse as σ√T grows")
display(gap.style.format({"σ√T": "{:.3f}", "Delta N(d1)": "{:.3f}", "Q-prob N(d2)": "{:.3f}", "gap": "{:+.3f}"}))

아래 애니메이션은 경로를 하나씩 추가하면서 만기에 ITM으로 끝난 경로의 **비율**을 셉니다. 노란 실선 $N(d_2)$ 가 이론적 극한이고, 흰 점선은 delta $N(d_1)$ 입니다.
seed 42의 250개 경로에서는 우연히 52.4%가 나와 delta와 비슷해 보이지만, 다음 셀에서 보듯 **표본 노이즈**입니다.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm
from scipy.optimize import brentq

# --- 1. Parameters, IV & Delta Calculation ---
T_days = 15
T = T_days / 365.0  # Time to maturity in years
r = 0.04            # Risk-free rate (4%)
S0_nvda = 214.0
market_price = 11.00 # ATM Call price

# Black-Scholes Call Price Formula
def bs_call(S, K, T, r, vol):
    d1 = (np.log(S / K) + (r + 0.5 * vol**2) * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

# Root finder to back out Implied Volatility
def implied_volatility(price, S, K, T, r):
    objective = lambda vol: bs_call(S, K, T, r, vol) - price
    return brentq(objective, 1e-4, 5.0)

# 1a. Back out IV
iv_nvda = implied_volatility(market_price, S0_nvda, S0_nvda, T, r)

# 1b. Calculate ATM Delta N(d1) and the risk-neutral ITM probability N(d2)
d1_nvda = (np.log(S0_nvda / S0_nvda) + (r + 0.5 * iv_nvda**2) * T) / (iv_nvda * np.sqrt(T))
d2_nvda = d1_nvda - iv_nvda * np.sqrt(T)
delta_nvda = norm.cdf(d1_nvda)
p_itm_q = norm.cdf(d2_nvda)   # what the simulated ITM share converges to (paths use drift r)

print(f"Calculated NVDA IV: {iv_nvda:.2%}")
print(f"Calculated NVDA Delta N(d1): {delta_nvda:.4f}")
print(f"Risk-neutral P(ITM) N(d2):   {p_itm_q:.4f}")

# --- 2. Data Generation (Monte Carlo Matrix) ---
n_paths = 250
n_steps = T_days
dt = T / n_steps

np.random.seed(42) # Fixed seed for reproducible display
dates = pd.date_range(start=pd.Timestamp.today().normalize(), periods=n_steps+1, freq='D')

# Generate paths under Q: drift r, the same σ (= IV) for every path
paths_nvda = np.zeros((n_steps + 1, n_paths))
paths_nvda[0] = S0_nvda
for i in range(1, n_steps + 1):
    Z = np.random.standard_normal(n_paths)
    paths_nvda[i] = paths_nvda[i-1] * np.exp((r - 0.5 * iv_nvda**2) * dt + iv_nvda * np.sqrt(dt) * Z)

# Pre-calculate ITM/OTM outcomes for the bar chart
final_prices = paths_nvda[-1, :]
is_itm = final_prices > S0_nvda

# Colors and randomized opacities
off_white = "#e0e0e0"
nvda_color = "#00ff88"  # Green for ITM
otm_color = "#ff4b4b"   # Red for OTM
opacities = np.random.uniform(0.15, 0.70, n_paths)

# Static limits to prevent axis jitter
y_min_nvda, y_max_nvda = np.min(paths_nvda) * 0.95, np.max(paths_nvda) * 1.05

# --- 3. Figure Setup ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"NVDA Paths under Q (σ = IV = {iv_nvda:.1%})", 
        "Share of Paths ITM at Maturity"
    ),
    horizontal_spacing=0.10
)

# --- Base Traces (Strict Order Required for Frames) ---
# Trace 0: Strike line (Left)
fig.add_trace(go.Scatter(x=[dates[0], dates[-1]], y=[S0_nvda, S0_nvda], line=dict(color=off_white, width=1, dash='dash'), opacity=0.5, name="Strike"), row=1, col=1)

# Traces 1 to n_paths: Empty placeholders for the paths to be drawn into
for i in range(n_paths):
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', line=dict(color=nvda_color, width=1.5), opacity=opacities[i], showlegend=False), row=1, col=1)

# Trace n_paths+1: Empty bar chart (share of paths, not counts)
fig.add_trace(go.Bar(
    x=["ITM (S_T > K)", "OTM (S_T ≤ K)"], 
    y=[0, 0], 
    marker_color=[nvda_color, otm_color], 
    text=["0%", "0%"], 
    textposition="inside", insidetextanchor="middle",  # keep labels clear of the N(d1)/N(d2) lines
    textfont=dict(color="black", size=14, weight="bold"),
    showlegend=False
), row=1, col=2)


# --- 4. Animation Frames & Slider Steps ---
frames = []
slider_steps = []

# Iterate Path-by-Path (not step-by-step)
for k in range(n_paths):
    frame_data = []
    
    # 0: Keep static strike line
    frame_data.append(go.Scatter(x=[dates[0], dates[-1]], y=[S0_nvda, S0_nvda]))
    
    # 1..n_paths: Render paths up to 'k', leave the rest empty
    for i in range(n_paths):
        if i <= k:
            frame_data.append(go.Scatter(x=dates, y=paths_nvda[:, i]))
        else:
            frame_data.append(go.Scatter(x=[None], y=[None]))
            
    # n_paths+1: Update Bar Chart tally based on the subset of paths drawn so far
    itm_count = np.sum(is_itm[:k+1])
    otm_count = (k + 1) - itm_count
    itm_pct = itm_count / (k + 1)
    otm_pct = otm_count / (k + 1)
    
    frame_data.append(go.Bar(
        x=["ITM (S_T > K)", "OTM (S_T ≤ K)"], 
        y=[itm_pct, otm_pct], 
        text=[f"{itm_pct:.1%} ({itm_count})", f"{otm_pct:.1%} ({otm_count})"]
    ))

    frame_name = f"f{k}"
    frames.append(go.Frame(data=frame_data, name=frame_name))
    
    step = {
        "args": [[frame_name], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}],
        "label": str(k + 1), # 1-indexed for the slider
        "method": "animate"
    }
    slider_steps.append(step)

fig.frames = frames

# --- 5. Layout with Buttons and Slider ---
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=600, width=1200,
    margin=dict(t=100, b=150, l=50, r=50), 
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [None, {"frame": {"duration": 200, "redraw": False}, "fromcurrent": True}] 
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Simulated Paths: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

axis_style = dict(
    showgrid=True, gridcolor='rgba(255,255,255,0.1)',
    tickfont=dict(color=off_white), linecolor=off_white,
    zeroline=False, title_font=dict(color=off_white)
)

# Left Plot: NVDA Price Paths
fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="Date", row=1, col=1)
fig.update_yaxes(axis_style, range=[y_min_nvda, y_max_nvda], title_text="NVDA Price ($)", row=1, col=1)

# Right Plot: Bar Chart (share of paths) with the two reference probabilities
fig.update_xaxes(axis_style, title_text="Outcome Status at Maturity", row=1, col=2)
fig.update_yaxes(axis_style, range=[0, 1], tickformat=".0%", title_text="Share of Simulated Paths", row=1, col=2)
fig.add_hline(y=p_itm_q, line=dict(color="#ffd166", width=2), row=1, col=2,
              annotation_text=f"N(d2) = {p_itm_q:.1%} (risk-neutral P(ITM), the true limit)", annotation_position="top left",
              annotation_font_color="#ffd166")
fig.add_hline(y=delta_nvda, line=dict(color="#e0e0e0", width=1.5, dash="dash"), row=1, col=2,
              annotation_text=f"Delta N(d1) = {delta_nvda:.1%}", annotation_position="top right",
              annotation_font_color="#e0e0e0")

fig.show()

##### 경로를 늘리면 어디로 수렴하나? 그리고 250개로 52.4%가 나온 건 얼마나 흔한 일인가?

경로 $n$ 개 중 ITM 개수는 이항분포 $\text{Bin}(n, p)$, $p = N(d_2)$ 를 따릅니다. 그래서 ITM 비율의 표준오차는

$$\text{SE} = \sqrt{\frac{p(1-p)}{n}}, \qquad n=250 \Rightarrow \sqrt{\frac{0.48 \times 0.52}{250}} \approx 3.2\%p$$

52.4%는 $N(d_2)=48.0\%$ 에서 약 $+1.4$ 표준오차 떨어진 값이라 충분히 흔하게 나옵니다. 250개로는 48%(=$N(d_2)$)와 53%(=delta)를 **구분할 수 없고**, 경로를 훨씬 많이 돌려야 구분됩니다.

In [ ]:
# ============================================================
# Convergence: the ITM share → N(d2), not N(d1)
# ============================================================
p_q = norm.cdf(d2)          # risk-neutral P(ITM)
delta = norm.cdf(d1)        # Delta

# (a) One long run of 1,000,000 paths. Only S_T matters, so draw it in one step with Z*.
N_big = 1_000_000
rng = np.random.default_rng(123)
S_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * rng.standard_normal(N_big))
running = np.cumsum(S_T > K) / np.arange(1, N_big + 1)
idx = np.unique(np.logspace(0, np.log10(N_big), 600).astype(int)) - 1
n_axis = idx + 1
band = 1.96 * np.sqrt(p_q * (1 - p_q) / n_axis)

# (b) 20,000 independent repeats of the notebook's experiment (250 paths each)
n_exp, n_per = 20_000, 250
shares = rng.binomial(n_per, p_q, size=n_exp) / n_per
observed = 0.524
z_obs = (observed - p_q) / np.sqrt(p_q * (1 - p_q) / n_per)

print(f"N(d2) risk-neutral P(ITM) = {p_q:.4f}    Delta N(d1) = {delta:.4f}")
for n in (250, 1_000, 10_000, 100_000, 1_000_000):
    print(f"  ITM share after {n:>9,} paths: {running[n - 1]:.4f}   (±{1.96 * np.sqrt(p_q * (1 - p_q) / n):.4f} 95% band)")
print(f"\nStd error with 250 paths = {np.sqrt(p_q * (1 - p_q) / n_per):.4f}")
print(f"52.4% is {z_obs:+.2f} std errors above N(d2); share of 250-path experiments with ≥ 52.4% ITM: {(shares >= observed).mean():.1%}")

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
                    subplot_titles=("Running ITM share (1,000,000 paths)", "ITM share in 20,000 repeats of a 250-path experiment"))
fig.add_trace(go.Scatter(x=np.concatenate([n_axis, n_axis[::-1]]),
                         y=np.concatenate([p_q + band, (p_q - band)[::-1]]),
                         fill='toself', fillcolor='rgba(255,209,102,0.15)', line=dict(width=0),
                         name="95% band around N(d2)", hoverinfo='skip'), row=1, col=1)
fig.add_trace(go.Scatter(x=n_axis, y=running[idx], mode='lines', line=dict(color="#00ff88", width=2),
                         name="simulated ITM share"), row=1, col=1)
fig.add_trace(go.Histogram(x=shares, xbins=dict(start=0.3 - 0.002, end=0.7, size=1 / n_per),
                           marker_color="#00d1ff", opacity=0.8, name="250-path experiments"), row=1, col=2)
for col in (1, 2):
    kw = dict(row=1, col=col)
    line_fn = fig.add_hline if col == 1 else fig.add_vline
    line_fn(p_q, line=dict(color="#ffd166", width=2), annotation_text=f"N(d2) = {p_q:.1%}", annotation_font_color="#ffd166",
            annotation_position="top right" if col == 1 else "top left", **kw)
    line_fn(delta, line=dict(color="#e0e0e0", width=1.5, dash='dash'), annotation_text=f"Delta N(d1) = {delta:.1%}",
            annotation_position="bottom right" if col == 1 else "top right", **kw)
fig.add_vline(observed, line=dict(color="#ff4b4b", width=1.5, dash='dot'), annotation_text="seed 42: 52.4%",
              annotation_position="right", annotation_font_color="#ff4b4b", row=1, col=2)
fig.update_xaxes(type="log", title_text="Number of simulated paths", row=1, col=1)
fig.update_yaxes(range=[0.3, 0.7], tickformat=".0%", title_text="Share ITM", row=1, col=1)
fig.update_xaxes(tickformat=".0%", title_text="Share of the 250 paths ending ITM", row=1, col=2)
fig.update_yaxes(title_text="Number of experiments", row=1, col=2)
fig.update_layout(template="plotly_dark", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                  height=520, width=1200, legend=dict(x=0.01, y=0.02, bgcolor='rgba(0,0,0,0)'), bargap=0.05)
fig.show()

###### ______________________________________________________________________________________________________________________________________

#### 📝 정리

1. **Implied vol**은 옵션 **가격** 하나를 BS 공식에 넣어 거꾸로 푼 σ입니다. 옵션 하나당 숫자 하나이고, 경로와는 무관합니다.
2. **경로 생성**은 위험중립 GBM $S_{t+\Delta t} = S_t\exp\big((r-\tfrac12\sigma^2)\Delta t + \sigma\sqrt{\Delta t}Z\big)$ 입니다. σ는 모든 경로·모든 시점에서 IV 하나이고, 경로를 다르게 만드는 건 **난수 $Z$ 뿐**입니다.
3. 경로마다 다르게 보이는 변동성은 **realized vol**(출력값)이고, 표본이 적어서 생기는 추정오차입니다. 스텝을 늘리면 모두 σ로 모입니다.
4. **Delta $=N(d_1)$** 는 헤지 비율입니다. 주식측도에서의 ITM 확률과 수식이 같을 뿐입니다.
5. drift $r$ 로 돌린 시뮬레이션의 ITM 비율은 **$N(d_2)$** (위험중립 확률)로 수렴합니다. 이 예제에서는 48.0%이고, delta 53.1%가 아닙니다.
6. 실제로 돈을 벌 확률은 실제 기대수익률 $\mu$ 에 달려 있고, 옵션 가격만으로는 알 수 없습니다.

---

#### 2.) 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary**

  - We explored the Black-Scholes model to build intuition about its applications and limitations in the real world, emphasizing that it is not intended as a forecasting or prediction model for asset prices or option outcomes.
  - A key takeaway is that Black-Scholes option prices between different contexts are not always apples-to-apples; differences in inputs, market assumptions, and volatility surfaces mean prices can't be universally compared, unlike implied volatility (iVol), which offers a consistent, relative scale for comparing options across strikes and maturities.
  - We clarified that while the Black-Scholes “delta” is often associated with probability, in reality it is a hedge ratio—it does not represent the actual chance of finishing in-the-money, but instead indicates how much the option price changes with small moves in the underlying.
  - Through practical examples and visualizations, we highlighted why it's crucial for traders to interpret model outputs with caution and always consider the assumptions, including market dynamics and model risk, when applying Black-Scholes-based insights to real-world option trading.

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$